In [ ]:
! pip install pypsa highspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.8/116.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.6 MB/s eta 0:00:00


## Step 1: Initialize Network and Carriers

In [ ]:
import pypsa
import pandas as pd

# 1. Initialize Network
n = pypsa.Network()
n.set_snapshots(range(4)) # 4 hours for a simple test

# 2. Add Energy Carriers
carriers = ["AC", "H2", "biogas", "natural_gas", "CO2", "SNG", "e-Kerosene", "e-Diesel"]
for c in carriers:
    n.add("Carrier", c)

## Step 2: Define the Buses (Nodes)

In [ ]:
# Electricity Market Bus
n.add("Bus", "electricity_market", carrier="AC")

# SRES and DRES local electrical buses (before conversion)
n.add("Bus", "SRES_AC", carrier="AC")
n.add("Bus", "DRES_AC", carrier="AC")

# Hydrogen Zones
n.add("Bus", "zone1_H2", carrier="H2")
n.add("Bus", "zone2_H2", carrier="H2") # YY00 region

# Derivative & CO2 Buses
n.add("Bus", "biogenic_CO2", carrier="CO2")
n.add("Bus", "SNG_bus", carrier="SNG")
n.add("Bus", "eKerosene_bus", carrier="e-Kerosene")
n.add("Bus", "eDiesel_bus", carrier="e-Diesel")

## Step 3: The Electricity Market & Renewables (SRES/DRES)

In [ ]:
# Electricity Market Generation & Demand
n.add("Generator", "grid_generation", bus="electricity_market",
      carrier="AC", p_nom=1000, marginal_cost=50)
n.add("Load", "grid_demand", bus="electricity_market", p_set=[400, 450, 500, 400])

# SRES (Shared RES) - Can send power to grid OR to its dedicated P2G
n.add("Generator", "SRES_plant", bus="SRES_AC", carrier="AC",
      p_nom=200, p_max_pu=[0.8, 0.6, 0.9, 0.5])
# Link allowing SRES to sell to the Electricity Market
n.add("Link", "SRES_to_grid", bus0="SRES_AC", bus1="electricity_market", p_nom_extendable=True)

# DRES (Dedicated RES) - Totally isolated from the main electricity market
n.add("Generator", "DRES_plant", bus="DRES_AC", carrier="AC",
      p_nom=300, p_max_pu=[0.9, 0.7, 0.8, 0.4])

## Step 4: Power-to-Gas (P2G) Infrastructure

In [ ]:
# 1. P2G from Electricity Market to Zone 1
n.add("Link", "P2G_grid_to_zone1", bus0="electricity_market", bus1="zone1_H2",
      efficiency=0.7, p_nom=100)

# 2. P2G from SRES to Zone 1
n.add("Link", "P2G_SRES_to_zone1", bus0="SRES_AC", bus1="zone1_H2",
      efficiency=0.7, p_nom=150)

# 3. P2G from Electricity Market to Zone 2
n.add("Link", "P2G_grid_to_zone2", bus0="electricity_market", bus1="zone2_H2",
      efficiency=0.7, p_nom=100)

# 4. P2G from DRES to Zone 2 (DRES ONLY goes here)
n.add("Link", "P2G_DRES_to_zone2", bus0="DRES_AC", bus1="zone2_H2",
      efficiency=0.7, p_nom=300)

# (Optional) Re-electrification: Arrow from Zone 2 back to Electricity Market
n.add("Link", "H2_to_grid", bus0="zone2_H2", bus1="electricity_market",
      efficiency=0.5, p_nom_extendable=True)

## Step 5: Zone 1 & Zone 2 Specifics (SMR, Storage, Demands)

In [ ]:
# --- Zone 1 ---
# SMR (b) - Biogas SMR
n.add("Generator", "SMR_biogas", bus="zone1_H2", carrier="H2", p_nom=50, marginal_cost=80)
# SMR (g) - Natural Gas SMR
n.add("Generator", "SMR_natgas", bus="zone1_H2", carrier="H2", p_nom=50, marginal_cost=60)

# Zone 1 Demand
n.add("Load", "zone1_demand", bus="zone1_H2", p_set=[50, 60, 50, 70])

# Zone 1 Storage (Tank & Salt Cavern Storage)
n.add("Store", "zone1_tank", bus="zone1_H2", e_nom=200)
n.add("Store", "zone1_cavern", bus="zone1_H2", e_nom=1000)

# --- Zone 2 ---
# Zone 2 (YY00) Storage
n.add("Store", "zone2_YY00_storage", bus="zone2_H2", e_nom=5000)

# --- Cross-Zone Interconnection ---
# Bi-directional arrow between Zone 1 and Zone 2
n.add("Link", "pipeline_z1_to_z2", bus0="zone1_H2", bus1="zone2_H2", p_nom=200)
n.add("Link", "pipeline_z2_to_z1", bus0="zone2_H2", bus1="zone1_H2", p_nom=200)

## Step 6: Downstream E-Fuels (Sector Coupling)

In [ ]:
# Supply of Biogenic CO2
n.add("Generator", "CO2_supply", bus="biogenic_CO2", carrier="CO2",
      p_nom_extendable=True, marginal_cost=20)

# 1. Methanation (H2 + CO2 -> SNG)
n.add("Link", "H2_to_SNG",
      bus0="zone2_H2",     # Input 1: Hydrogen
      bus1="SNG_bus",      # Output: SNG
      bus2="biogenic_CO2", # Input 2: CO2
      efficiency=0.6,      # Conversion efficiency H2 -> SNG
      efficiency2=-0.2,    # Negative means it *consumes* CO2 per unit of H2 input
      p_nom_extendable=True)

# 2. Fischer-Tropsch / Synthesis (H2 + CO2 -> e-Kerosene)
n.add("Link", "H2_to_eKerosene",
      bus0="zone2_H2", bus1="eKerosene_bus", bus2="biogenic_CO2",
      efficiency=0.5, efficiency2=-0.3, p_nom_extendable=True)

# 3. Fischer-Tropsch / Synthesis (H2 + CO2 -> E-Diesel)
n.add("Link", "H2_to_eDiesel",
      bus0="zone2_H2", bus1="eDiesel_bus", bus2="biogenic_CO2",
      efficiency=0.5, efficiency2=-0.3, p_nom_extendable=True)

# Add final demands for the derivatives
n.add("Load", "SNG_demand", bus="SNG_bus", p_set=[20, 20, 20, 20])
n.add("Load", "eKerosene_demand", bus="eKerosene_bus", p_set=[10, 10, 10, 10])
n.add("Load", "eDiesel_demand", bus="eDiesel_bus", p_set=[15, 15, 15, 15])

## Step 7: Solve the Model


In [ ]:
# Run the Linear Optimal Power Flow (LOPF)
status, condition = n.optimize(solver_name='highs') # or 'gurobi', 'cbc', 'highs'

print(f"Optimization Status: {status}")
print(f"Total System Cost: €{n.objective:.2f}")

# Example: Check how much Hydrogen was produced by the DRES P2G
print("\nDRES P2G Production (MW):")
print(n.links_t.p1["P2G_DRES_to_zone2"])

/tmp/ipykernel_525/2145655851.py:2: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.



Optimization Status: ok
Total System Cost: €60616.67

DRES P2G Production (MW):
snapshot
0   -189.0
1   -147.0
2   -168.0
3    -84.0
Name: P2G_DRES_to_zone2, dtype: float64
